# IOU Scoring & Linear Interpolation Experiments
### Part 1: IOU scores for 71 programs (70 golden + uniform_lower_diagonal) across BERT, GPT-2, TinyLlama
### Part 2: Linear interpolation with k=1..20 top-fitting programs per head

**Outputs saved to `data/`:**
- `data/iou_scores_bert.csv`
- `data/iou_scores_gpt2.csv`
- `data/iou_scores_tinyllama.csv`
- `data/interpolation_k_bert.csv`
- `data/interpolation_k_gpt2.csv`
- `data/interpolation_k_tinyllama.csv`


In [ ]:
# Install dependencies if needed
# !pip install torch transformers spacy nltk tqdm scikit-learn
# !python -m spacy download en_core_web_sm

In [2]:
import os
import csv
import json
import time
import traceback
import warnings
import inspect
import random
from pathlib import Path
from typing import Callable, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import spacy
from tqdm import tqdm
from transformers import (
    AutoTokenizer, AutoModel,
    PreTrainedModel, PreTrainedTokenizerBase
)
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

# reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

nlp = spacy.load("en_core_web_sm")

os.makedirs("data", exist_ok=True)
print("Imports OK.")


Imports OK.


In [3]:
# ── Scoring helpers ──────────────────────────────────────────────────────────

def iou_score(p: np.ndarray, q: np.ndarray) -> float:
    """IoU similarity between two attention distributions (higher = better)."""
    p = np.clip(p, 1e-12, 1.0)
    q = np.clip(q, 1e-12, 1.0)
    intersection = np.sum(np.minimum(p, q))
    union        = np.sum(np.maximum(p, q))
    return float(intersection / union) if union > 0 else 0.0


def score_head_iou(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    layer: int,
    head: int,
    pattern: Callable,
    sentence: str,
) -> float:
    """
    Returns mean row-wise IoU between actual attention and predicted pattern.
    Returns NaN on any error so callers can handle gracefully.
    """
    try:
        tokens = tokenizer(sentence, return_tensors="pt")
        with torch.no_grad():
            att = model(**tokens, output_attentions=True).attentions[layer][0, head].detach().numpy()

        # detect paradigm: 1-arg (Jacob's) vs 2-arg (sentence, tokenizer)
        n_params = len(inspect.signature(pattern).parameters)
        if n_params == 1:
            _, pred_att = pattern(sentence)
        else:
            _, pred_att = pattern(sentence, tokenizer)

        pred_att = np.array(pred_att, dtype=float)
        min_len = min(att.shape[0], pred_att.shape[0])
        att      = att[:min_len, :min_len]
        pred_att = pred_att[:min_len, :min_len]

        row_ious = [iou_score(att[i], pred_att[i]) for i in range(min_len)]
        return float(np.mean(row_ious))

    except Exception:
        return float("nan")


In [4]:
# ── Load golden programs + uniform_lower_diagonal ────────────────────────────
import sys
sys.path.insert(0, ".")

import golden_programs
from programs import *  # brings named imports into scope if needed

all_programs: List[Callable] = [
    obj for name, obj in inspect.getmembers(golden_programs, inspect.isfunction)
]

# ---- uniform_lower_diagonal ------------------------------------------------
def uniform_lower_diagonal(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    toks = tokenizer([sentence], return_tensors="pt")
    n = len(toks.input_ids[0])
    out = np.full((n, n), 1e-4)
    for i in range(n):
        for j in range(n):
            if j <= i:
                out[i, j] = 1.0
    out += 1e-4
    out = out / out.sum(axis=1, keepdims=True)
    return "Uniform Lower Diagonal Attention Pattern", out

all_programs.append(uniform_lower_diagonal)
# ---------------------------------------------------------------------------

# Save index mapping so CSVs (which store program_idx) are interpretable later
PROGRAM_INDEX_MAP = {i: p.__name__ for i, p in enumerate(all_programs)}
print(f"Total programs loaded: {len(all_programs)}")
print("Index -> program name mapping (also saved to data/program_index_map.json):")
for idx, name in PROGRAM_INDEX_MAP.items():
    print(f"  [{idx:2d}] {name}")

import json as _json
with open("data/program_index_map.json", "w") as _f:
    _json.dump(PROGRAM_INDEX_MAP, _f, indent=2)


Total programs loaded: 71
Index -> program name mapping (also saved to data/program_index_map.json):
  [ 0] adverbial_modulation
  [ 1] appositive_phrase_attention
  [ 2] cls_attention
  [ 3] complement_adjunct_relationship
  [ 4] compound_element_association
  [ 5] compound_modifier_attention
  [ 6] compound_word_attention_pattern
  [ 7] conjunction_based_grouping
  [ 8] conjunction_resolution
  [ 9] contextual_anchoring
  [10] coord_and_verb_dependency
  [11] coordination_attention
  [12] coreference_resolution
  [13] dependencies
  [14] dominant_subject_attention
  [15] emphasize_verbs_and_objects
  [16] eos_attention
  [17] first_token_dominance
  [18] first_token_domination
  [19] first_token_emphasis
  [20] head_initial_token_emphasis
  [21] high_saliency_relationship_detection
  [22] initial_contextual_attention
  [23] initial_element_reinforcement
  [24] initial_phrase_contextualization
  [25] initial_phrase_dominance
  [26] initial_reference_attention
  [27] initial_token_anch

In [5]:
# ── Load generic_sentences and sample 5 (randomized) ────────────────────────
df_json = pd.read_json("data/generic_sentences.json")
generic_sentences_all = df_json[0].tolist()

# Shuffle with fixed seed then take first 5 — clearly randomized
_rng = random.Random(RANDOM_SEED)
_shuffled = generic_sentences_all[:]
_rng.shuffle(_shuffled)
SAMPLE_SENTENCES: List[str] = _shuffled[:5]

# Save the mapping so sentence_idx in CSVs is always recoverable
SENTENCE_INDEX_MAP = {i: s for i, s in enumerate(SAMPLE_SENTENCES)}
print(f"Total generic sentences available: {len(generic_sentences_all)}")
print("Sampled 5 sentences (sentence_idx -> text):")
for i, s in SENTENCE_INDEX_MAP.items():
    print(f"  [{i}] {s}")

import json as _json
with open("data/sentence_index_map.json", "w") as _f:
    _json.dump(SENTENCE_INDEX_MAP, _f, indent=2)


Total generic sentences available: 100
Sampled 5 sentences (sentence_idx -> text):
  [0] The rhythmic crash of the waves, breaking against the shore, provided a soothing soundtrack to the quiet evening.
  [1] He contemplated, for a moment, the vastness of the universe, its endless possibilities, and his tiny place within it.
  [2] 'That's an excellent point!' she agreed, nodding enthusiastically, acknowledging the validity of his argument.
  [3] If you truly want to succeed, you must work diligently, persevere through challenges, and never give up on your dreams.
  [4] He meticulously organized his tools: wrenches, screwdrivers, pliers, and a tape measure, all in their designated spots.


In [6]:
# ── Model configurations ─────────────────────────────────────────────────────
MODEL_CONFIGS = {
    # "bert": {
    #     "model_name": "bert-base-uncased",
    #     "out_csv":    "data/iou_scores_bert.csv",
    # },
    # "gpt2": {
    #     "model_name": "openai-community/gpt2",
    #     "out_csv":    "data/iou_scores_gpt2.csv",
    # },
    "tinyllama": {
        "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
        "out_csv":    "data/iou_scores_tinyllama.csv",
    },
}


In [ ]:
# ── PART 1: IOU scoring — streams every row to CSV immediately ───────────────
#
# CSV columns: model_key, layer, head, program_idx, sentence_idx, iou_score
#   program_idx  ->  all_programs[i].__name__  (see data/program_index_map.json)
#   sentence_idx ->  SAMPLE_SENTENCES[i]       (see data/sentence_index_map.json)
#
# Resume-safe: already-computed (layer, head, program_idx, sentence_idx) rows
# are skipped so the cell is safe to re-run after interruption.

PART1_COLUMNS = ["layer", "head", "program_idx", "sentence_idx", "iou_score"]


def load_done_keys(csv_path: str) -> set:
    """Return set of (layer, head, program_idx, sentence_idx) already in the CSV."""
    done = set()
    if not os.path.exists(csv_path):
        return done
    try:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            done.add((int(row["layer"]), int(row["head"]), int(row["program_idx"]), int(row["sentence_idx"])))
    except Exception as e:
        print(f"  [warn] Could not read existing CSV {csv_path}: {e}")
    return done


for model_key, cfg in MODEL_CONFIGS.items():
    model_name = cfg["model_name"]
    out_csv    = cfg["out_csv"]

    print(f"\n{'='*60}")
    print(f"  Model: {model_key}  ({model_name})")
    print(f"  Output: {out_csv}")
    print(f"{'='*60}")

    # ── load model ────────────────────────────────────────────────────────────
    try:
        model_obj = AutoModel.from_pretrained(model_name, output_attentions=True)
        tokenizer_obj = AutoTokenizer.from_pretrained(model_name)
        model_obj.eval()
    except Exception as e:
        print(f"  [ERROR] Could not load model {model_name}: {e}")
        traceback.print_exc()
        continue

    num_layers = model_obj.config.num_hidden_layers
    num_heads  = model_obj.config.num_attention_heads
    print(f"  Layers: {num_layers}  Heads: {num_heads}  Programs: {len(all_programs)}")

    # ── resume support ────────────────────────────────────────────────────────
    done_keys = load_done_keys(out_csv)
    print(f"  Already done rows: {len(done_keys)}")

    write_header = not os.path.exists(out_csv) or os.path.getsize(out_csv) == 0
    errors = 0

    with open(out_csv, "a", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        if write_header:
            writer.writerow(PART1_COLUMNS)
            fh.flush()

        total = num_layers * num_heads * len(all_programs) * len(SAMPLE_SENTENCES)
        done_count = len(done_keys)
        t0 = time.time()

        for p_idx, prog in enumerate(all_programs):
            for layer in range(num_layers):
                for head in range(num_heads):
                    for s_idx, sentence in enumerate(SAMPLE_SENTENCES):
                        key = (layer, head, p_idx, s_idx)
                        if key in done_keys:
                            done_count += 1
                            continue

                        score = score_head_iou(model_obj, tokenizer_obj, layer, head, prog, sentence)
                        score_out = round(score, 3) if not np.isnan(score) else score
                        writer.writerow([layer, head, p_idx, s_idx, score_out])
                        fh.flush()  # stream to disk immediately
                        done_count += 1

                        if np.isnan(score):
                            errors += 1

                        # lightweight progress every 1,000 rows
                        if done_count % 720 == 0:
                            elapsed = time.time() - t0
                            pct = done_count / total * 100
                            eta = elapsed / done_count * (total - done_count) if done_count else 0
                            print(f"  [{model_key}] {done_count}/{total} ({pct:.1f}%) | "
                                  f"elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | errors {errors}")

    # clean up to free VRAM/RAM before next model
    del model_obj, tokenizer_obj
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    print(f"  DONE {model_key}. Errors (NaN): {errors}")
    df_check = pd.read_csv(out_csv)
    print(f"  Rows saved: {len(df_check)}")

print("\n[PART 1 COMPLETE]")



  Model: bert  (bert-base-uncased)
  Output: data/iou_scores_bert.csv
  Layers: 12  Heads: 12  Programs: 71
  Already done rows: 0
  [bert] 720/51120 (1.4%) | elapsed 0.6m | ETA 42.1m | errors 0
  [bert] 1440/51120 (2.8%) | elapsed 1.1m | ETA 38.7m | errors 0
  [bert] 2160/51120 (4.2%) | elapsed 2.0m | ETA 44.8m | errors 0
  [bert] 2880/51120 (5.6%) | elapsed 2.9m | ETA 49.0m | errors 0
  [bert] 3600/51120 (7.0%) | elapsed 4.2m | ETA 55.5m | errors 0
  [bert] 4320/51120 (8.5%) | elapsed 11.8m | ETA 127.8m | errors 0
  [bert] 5040/51120 (9.9%) | elapsed 12.3m | ETA 112.6m | errors 0
  [bert] 5760/51120 (11.3%) | elapsed 12.9m | ETA 101.8m | errors 0
  [bert] 6480/51120 (12.7%) | elapsed 13.5m | ETA 93.0m | errors 0
  [bert] 7200/51120 (14.1%) | elapsed 14.0m | ETA 85.5m | errors 0
  [bert] 7920/51120 (15.5%) | elapsed 14.5m | ETA 79.2m | errors 0
  [bert] 8640/51120 (16.9%) | elapsed 15.0m | ETA 74.0m | errors 0
  [bert] 9360/51120 (18.3%) | elapsed 15.6m | ETA 69.8m | errors 0
  [bert

In [7]:
# ── Resume TinyLlama IOU scoring — last ~33%, saves to separate CSV ───────────
import os, csv, time, traceback
import numpy as np
import pandas as pd
from transformers import AutoModel, AutoTokenizer

TINYLLAMA_ORIGINAL_CSV = "data/iou_scores_tinyllama.csv"
TINYLLAMA_RESUME_CSV   = "data/iou_scores_tinyllama_part2.csv"

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# ── load model ────────────────────────────────────────────────────────────────
try:
    model_obj     = AutoModel.from_pretrained(model_name, output_attentions=True)
    tokenizer_obj = AutoTokenizer.from_pretrained(model_name)
    model_obj.eval()
except Exception as e:
    print(f"[ERROR] Could not load model: {e}")
    traceback.print_exc()
    raise

num_layers = model_obj.config.num_hidden_layers
num_heads  = model_obj.config.num_attention_heads
print(f"Layers: {num_layers}  Heads: {num_heads}  Programs: {len(all_programs)}")

# ── build done_keys from BOTH csvs so nothing is re-computed ─────────────────
def load_done_keys_from(csv_path):
    done = set()
    if not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0:
        return done
    try:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            done.add((int(row["layer"]), int(row["head"]),
                      int(row["program_idx"]), int(row["sentence_idx"])))
    except Exception as e:
        print(f"  [warn] Could not read {csv_path}: {e}")
    return done

done_keys  = load_done_keys_from(TINYLLAMA_ORIGINAL_CSV)
done_keys |= load_done_keys_from(TINYLLAMA_RESUME_CSV)
print(f"Already done rows (both CSVs): {len(done_keys):,}")

PART1_COLUMNS  = ["layer", "head", "program_idx", "sentence_idx", "iou_score"]
total          = num_layers * num_heads * len(all_programs) * len(SAMPLE_SENTENCES)
remaining      = total - len(done_keys)
write_header   = not os.path.exists(TINYLLAMA_RESUME_CSV) or os.path.getsize(TINYLLAMA_RESUME_CSV) == 0
errors         = 0
done_count     = len(done_keys)
t0             = time.time()

print(f"Total: {total:,}  |  Remaining: {remaining:,}  ({remaining/total*100:.1f}%)")

with open(TINYLLAMA_RESUME_CSV, "a", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    if write_header:
        writer.writerow(PART1_COLUMNS)
        fh.flush()

    for p_idx, prog in enumerate(all_programs):
        for layer in range(num_layers):
            for head in range(num_heads):
                for s_idx, sentence in enumerate(SAMPLE_SENTENCES):
                    key = (layer, head, p_idx, s_idx)
                    if key in done_keys:
                        done_count += 1
                        continue

                    score     = score_head_iou(model_obj, tokenizer_obj, layer, head, prog, sentence)
                    score_out = round(score, 3) if not np.isnan(score) else score
                    writer.writerow([layer, head, p_idx, s_idx, score_out])
                    fh.flush()
                    done_count += 1

                    if np.isnan(score):
                        errors += 1

                    if done_count % 720 == 0:
                        elapsed = time.time() - t0
                        pct     = done_count / total * 100
                        eta     = elapsed / max(done_count - len(done_keys), 1) * remaining
                        print(f"  [tinyllama] {done_count}/{total} ({pct:.1f}%) | "
                              f"elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | errors {errors}")

del model_obj, tokenizer_obj
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\nDONE. Errors: {errors}")
print(f"  Original CSV rows: {len(pd.read_csv(TINYLLAMA_ORIGINAL_CSV)):,}")
print(f"  Resume CSV rows:   {len(pd.read_csv(TINYLLAMA_RESUME_CSV)):,}")
print(f"  Combined total:    {len(done_keys) + errors:,} / {total:,}")

Layers: 22  Heads: 32  Programs: 71
Already done rows (both CSVs): 167,801
Total: 249,920  |  Remaining: 82,119  (32.9%)
  [tinyllama] 336240/249920 (134.5%) | elapsed 4.3m | ETA 2.1m | errors 0
  [tinyllama] 336960/249920 (134.8%) | elapsed 8.9m | ETA 4.3m | errors 0
  [tinyllama] 337680/249920 (135.1%) | elapsed 13.6m | ETA 6.6m | errors 0
  [tinyllama] 338400/249920 (135.4%) | elapsed 18.3m | ETA 8.8m | errors 0
  [tinyllama] 339120/249920 (135.7%) | elapsed 23.1m | ETA 11.1m | errors 0
  [tinyllama] 339840/249920 (136.0%) | elapsed 27.8m | ETA 13.3m | errors 0
  [tinyllama] 340560/249920 (136.3%) | elapsed 32.5m | ETA 15.4m | errors 0
  [tinyllama] 341280/249920 (136.6%) | elapsed 37.2m | ETA 17.6m | errors 0
  [tinyllama] 342000/249920 (136.8%) | elapsed 41.9m | ETA 19.7m | errors 0
  [tinyllama] 342720/249920 (137.1%) | elapsed 46.6m | ETA 21.9m | errors 0
  [tinyllama] 343440/249920 (137.4%) | elapsed 51.3m | ETA 24.0m | errors 0
  [tinyllama] 344160/249920 (137.7%) | elapsed 56

In [ ]:
# ── Part 1 verification ───────────────────────────────────────────────────────
for model_key, cfg in MODEL_CONFIGS.items():
    csv_path = cfg["out_csv"]
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f"{model_key}: {len(df)} rows | NaN iou: {df['iou_score'].isna().sum()} | "
              f"programs: {df['program_idx'].nunique()} | mean iou: {df['iou_score'].mean():.4f}")
    else:
        print(f"{model_key}: CSV not found at {csv_path}")


## Part 2: Linear Interpolation with k = 1 → 20

For **every head** and **every model**, fit a linear combination of the top-k programs 
(ranked by IoU from Part 1, per head, averaged across the 5 sentences). 
Sweep k from 1 to 20. Record the IoU of the fitted combination.

**CSV columns:** `model_key, layer, head, k, iou_score, program_idxs_used`


In [8]:
# ── Part 2 helpers ───────────────────────────────────────────────────────────

def linear_interp_iou(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    layer: int,
    head: int,
    top_k_programs: List[Callable],
    sentences: List[str],
) -> float:
    """
    Fit a linear combination of top_k_programs to minimize squared error vs
    actual attention, then measure mean IoU of the fitted combination.
    Returns NaN on failure.
    """
    all_ious = []

    for sentence in sentences:
        try:
            tokens = tokenizer(sentence, return_tensors="pt")
            with torch.no_grad():
                att = model(**tokens, output_attentions=True).attentions[layer][0, head].detach().numpy()

            # build feature matrix: each program -> flattened row
            X_rows = []
            for prog in top_k_programs:
                try:
                    n_params = len(inspect.signature(prog).parameters)
                    _, pred = prog(sentence) if n_params == 1 else prog(sentence, tokenizer)
                    pred = np.array(pred, dtype=float)
                    min_len = min(att.shape[0], pred.shape[0])
                    pred = pred[:min_len, :min_len]
                    X_rows.append(pred.flatten())
                except Exception:
                    return float("nan")

            min_len = min(att.shape[0], len(X_rows[0]) ** 0.5)  # crude guard
            seq_len = att.shape[0]

            # trim all rows to same length
            flat_len = seq_len * seq_len
            X_mat = np.array([r[:flat_len] for r in X_rows]).T          # (flat_len, k)
            y_vec = att.flatten()[:flat_len]

            X_mat = np.nan_to_num(X_mat)
            y_vec = np.nan_to_num(y_vec)

            reg = LinearRegression(fit_intercept=True).fit(X_mat, y_vec)
            fitted = reg.intercept_ + X_mat @ reg.coef_
            fitted = fitted.reshape(seq_len, seq_len)

            # normalize rows so they sum to 1 (like attention)
            row_sums = fitted.sum(axis=1, keepdims=True)
            row_sums = np.where(row_sums == 0, 1, row_sums)
            fitted = fitted / row_sums
            fitted = np.clip(fitted, 1e-12, None)

            row_ious = [iou_score(att[i], fitted[i]) for i in range(seq_len)]
            all_ious.append(float(np.mean(row_ious)))

        except Exception:
            all_ious.append(float("nan"))

    valid = [v for v in all_ious if not np.isnan(v)]
    return float(np.mean(valid)) if valid else float("nan")


K_VALUES = list(range(1, 21))   # 1, 2, ..., 20

PART2_COLUMNS = ["model_key", "layer", "head", "k", "iou_score", "program_idxs_used"]

print(f"K values: {K_VALUES}")
print(f"Part 2 columns: {PART2_COLUMNS}")


K values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Part 2 columns: ['model_key', 'layer', 'head', 'k', 'iou_score', 'program_idxs_used']


In [ ]:
# ── PART 2: Linear interpolation sweep ───────────────────────────────────────
#
# Strategy:
#   1. Load Part 1 CSV to rank programs per head by mean IoU (desc)
#   2. For each head, for k=1..20, take top-k programs and fit linear combo
#   3. Stream each row to CSV
#
# Resume-safe: skips (layer, head, k) already in the output CSV.

INTERP_CONFIGS = {
    # "bert":      "data/interpolation_k_bert.csv",
    # "gpt2":      "data/interpolation_k_gpt2.csv",
    "tinyllama": "data/interpolation_k_tinyllama.csv",
}


def load_done_interp(csv_path: str) -> set:
    done = set()
    if not os.path.exists(csv_path):
        return done
    try:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            done.add((int(row["layer"]), int(row["head"]), int(row["k"])))
    except Exception as e:
        print(f"  [warn] Could not read {csv_path}: {e}")
    return done


for model_key in MODEL_CONFIGS:
    iou_csv   = MODEL_CONFIGS[model_key]["out_csv"]
    out_csv   = INTERP_CONFIGS[model_key]
    model_name = MODEL_CONFIGS[model_key]["model_name"]

    print(f"\n{'='*60}")
    print(f"  Model: {model_key}  ({model_name})")
    print(f"  Reading IOU from: {iou_csv}")
    print(f"  Writing to:       {out_csv}")
    print(f"{'='*60}")

    # ── check Part 1 data ─────────────────────────────────────────────────────
    if not os.path.exists(iou_csv):
        print(f"  [SKIP] Part 1 CSV not found: {iou_csv}")
        continue

    iou_df = pd.read_csv(iou_csv)
    if iou_df.empty:
        print(f"  [SKIP] Part 1 CSV is empty.")
        continue

    # ── load model ────────────────────────────────────────────────────────────
    try:
        model_obj = AutoModel.from_pretrained(model_name, output_attentions=True)
        tokenizer_obj = AutoTokenizer.from_pretrained(model_name)
        model_obj.eval()
    except Exception as e:
        print(f"  [ERROR] Could not load model: {e}")
        traceback.print_exc()
        continue

    num_layers = model_obj.config.num_hidden_layers
    num_heads  = model_obj.config.num_attention_heads

    # ── build per-head program ranking from Part 1 ────────────────────────────
    # mean iou per (layer, head, program) across the 5 sentences
    head_rankings: dict = {}   # (layer, head) -> ordered list of Callable

    grouped = (
        iou_df.groupby(["layer", "head", "program_idx"])["iou_score"]
        .mean()
        .reset_index()
    )
    for (layer, head), grp in grouped.groupby(["layer", "head"]):
        sorted_grp = grp.sort_values("iou_score", ascending=False)
        ordered_fns = []
        for p_idx in sorted_grp["program_idx"]:
            fn = all_programs[int(p_idx)] if int(p_idx) < len(all_programs) else None
            if fn is not None:
                ordered_fns.append(fn)
        head_rankings[(int(layer), int(head))] = ordered_fns

    max_k = max(K_VALUES)
    total = num_layers * num_heads * len(K_VALUES)
    done_keys = load_done_interp(out_csv)
    print(f"  Already done rows: {len(done_keys)}  |  Total to do: {total}")

    write_header = not os.path.exists(out_csv) or os.path.getsize(out_csv) == 0
    errors = 0
    done_count = len(done_keys)
    t0 = time.time()

    with open(out_csv, "a", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        if write_header:
            writer.writerow(PART2_COLUMNS)
            fh.flush()

        for layer in range(num_layers):
            for head in range(num_heads):
                ranked_fns = head_rankings.get((layer, head), all_programs)

                for k in K_VALUES:
                    key = (layer, head, k)
                    if key in done_keys:
                        done_count += 1
                        continue

                    top_k_fns = ranked_fns[:k]
                    if not top_k_fns:
                        top_k_fns = all_programs[:k]

                    iou = linear_interp_iou(
                        model_obj, tokenizer_obj,
                        layer, head, top_k_fns, SAMPLE_SENTENCES,
                    )

                    # store indices, not names — use data/program_index_map.json to recover names
                    prog_idx_used = "|".join(str(all_programs.index(f)) for f in top_k_fns)
                    iou_out = round(iou, 6) if not np.isnan(iou) else iou
                    writer.writerow([model_key, layer, head, k, iou_out, prog_idx_used])
                    fh.flush()
                    done_count += 1

                    if np.isnan(iou):
                        errors += 1

                    if done_count % 200 == 0:
                        elapsed = time.time() - t0
                        pct = done_count / total * 100
                        eta = elapsed / done_count * (total - done_count) if done_count else 0
                        print(f"  [{model_key}] {done_count}/{total} ({pct:.1f}%) | "
                              f"elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | errors {errors}")

    del model_obj, tokenizer_obj
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    print(f"  DONE {model_key}.  Errors (NaN): {errors}")
    df_check = pd.read_csv(out_csv)
    print(f"  Rows saved: {len(df_check)}")

print("\n[PART 2 COMPLETE]")



  Model: tinyllama  (TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T)
  Reading IOU from: data/iou_scores_tinyllama.csv
  Writing to:       data/interpolation_k_tinyllama.csv
  Already done rows: 0  |  Total to do: 14080
  [tinyllama] 200/14080 (1.4%) | elapsed 13.1m | ETA 909.9m | errors 0
  [tinyllama] 400/14080 (2.8%) | elapsed 26.2m | ETA 896.1m | errors 0
  [tinyllama] 600/14080 (4.3%) | elapsed 39.1m | ETA 878.1m | errors 0
  [tinyllama] 800/14080 (5.7%) | elapsed 51.7m | ETA 857.6m | errors 0


In [9]:
# ── Resume TinyLlama interpolation from where it stopped ─────────────────────

iou_csv    = "data/iou_scores_tinyllama.csv"
out_csv    = "data/interpolation_k_tinyllama.csv"
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# merge both part1 CSVs for rankings
iou_df = pd.concat([
    pd.read_csv("data/iou_scores_tinyllama.csv"),
    pd.read_csv("data/iou_scores_tinyllama_part2.csv"),
]).drop_duplicates(["layer", "head", "program_idx", "sentence_idx"])
print(f"IOU rows loaded: {len(iou_df):,}")

try:
    model_obj     = AutoModel.from_pretrained(model_name, output_attentions=True)
    tokenizer_obj = AutoTokenizer.from_pretrained(model_name)
    model_obj.eval()
except Exception as e:
    print(f"[ERROR] {e}"); traceback.print_exc(); raise

num_layers = model_obj.config.num_hidden_layers
num_heads  = model_obj.config.num_attention_heads

# build per-head rankings
head_rankings = {}
grouped = (
    iou_df.groupby(["layer", "head", "program_idx"])["iou_score"]
    .mean().reset_index()
)
for (layer, head), grp in grouped.groupby(["layer", "head"]):
    sorted_grp  = grp.sort_values("iou_score", ascending=False)
    ordered_fns = [all_programs[int(p)] for p in sorted_grp["program_idx"]
                   if int(p) < len(all_programs)]
    head_rankings[(int(layer), int(head))] = ordered_fns

# resume: read existing done keys — handle model_key column if present
done_keys = set()
if os.path.exists(out_csv) and os.path.getsize(out_csv) > 0:
    try:
        existing = pd.read_csv(out_csv)
        # drop model_key column if it snuck in
        for _, row in existing.iterrows():
            done_keys.add((int(row["layer"]), int(row["head"]), int(row["k"])))
    except Exception as e:
        print(f"[warn] {e}")

total      = num_layers * num_heads * len(K_VALUES)
remaining  = total - len(done_keys)
done_count = len(done_keys)
errors     = 0
t0         = time.time()
print(f"Already done: {done_count:,} / {total:,} — remaining: {remaining:,}")

write_header = not os.path.exists(out_csv) or os.path.getsize(out_csv) == 0
COLUMNS = ["layer", "head", "k", "iou_score", "program_idxs_used"]

with open(out_csv, "a", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    if write_header:
        writer.writerow(COLUMNS)
        fh.flush()

    for layer in range(num_layers):
        for head in range(num_heads):
            ranked_fns = head_rankings.get((layer, head), all_programs)
            for k in K_VALUES:
                key = (layer, head, k)
                if key in done_keys:
                    done_count += 1
                    continue

                top_k_fns = ranked_fns[:k] if ranked_fns else all_programs[:k]
                iou       = linear_interp_iou(model_obj, tokenizer_obj,
                                              layer, head, top_k_fns, SAMPLE_SENTENCES)

                prog_idx_used = "|".join(str(all_programs.index(f)) for f in top_k_fns)
                iou_out       = round(iou, 3) if not np.isnan(iou) else iou
                writer.writerow([layer, head, k, iou_out, prog_idx_used])
                fh.flush()
                done_count += 1

                if np.isnan(iou):
                    errors += 1

                if done_count % 200 == 0:
                    new_done  = done_count - len(done_keys)
                    elapsed   = time.time() - t0
                    eta       = elapsed / max(new_done, 1)  * remaining
                    print(f"  [tinyllama] {done_count}/{total} ({done_count/total*100:.1f}%) | "
                          f"elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | errors {errors}")

del model_obj, tokenizer_obj
torch.cuda.empty_cache() if torch.cuda.is_available() else None

df_check = pd.read_csv(out_csv)
print(f"\nDONE. Rows in CSV: {len(df_check):,} / {total:,} | errors: {errors}")

IOU rows loaded: 249,920
Already done: 952 / 14,080 — remaining: 13,128
  [tinyllama] 2000/14080 (14.2%) | elapsed 8.9m | ETA 111.1m | errors 0
  [tinyllama] 2200/14080 (15.6%) | elapsed 28.9m | ETA 304.1m | errors 0
  [tinyllama] 2400/14080 (17.0%) | elapsed 49.0m | ETA 444.3m | errors 0
  [tinyllama] 2600/14080 (18.5%) | elapsed 68.5m | ETA 545.9m | errors 0
  [tinyllama] 2800/14080 (19.9%) | elapsed 89.6m | ETA 636.8m | errors 0
  [tinyllama] 3000/14080 (21.3%) | elapsed 105.7m | ETA 677.2m | errors 0
  [tinyllama] 3200/14080 (22.7%) | elapsed 117.8m | ETA 687.7m | errors 0
  [tinyllama] 3400/14080 (24.1%) | elapsed 129.3m | ETA 693.6m | errors 0
  [tinyllama] 3600/14080 (25.6%) | elapsed 141.0m | ETA 699.0m | errors 0
  [tinyllama] 3800/14080 (27.0%) | elapsed 153.4m | ETA 706.9m | errors 0
  [tinyllama] 4000/14080 (28.4%) | elapsed 165.0m | ETA 710.6m | errors 0
  [tinyllama] 4200/14080 (29.8%) | elapsed 176.5m | ETA 713.5m | errors 0
  [tinyllama] 4400/14080 (31.2%) | elapsed 188

In [11]:
# ── PART 2: Linear interpolation sweep ───────────────────────────────────────
#
# Strategy:
#   1. Load Part 1 CSV to rank programs per head by mean IoU (desc)
#   2. For each head, for k=1..20, take top-k programs and fit linear combo
#   3. Stream each row to CSV
#
# Resume-safe: skips (layer, head, k) already in the output CSV.

INTERP_CONFIGS = {
    # "bert":      "data/interpolation_k_bert.csv",
    # "gpt2":      "data/interpolation_k_gpt2.csv",
    "tinyllama": "data/interpolation_k_tinyllama_3.csv",
}


def load_done_interp(csv_path: str) -> set:
    done = set()
    if not os.path.exists(csv_path):
        return done
    try:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            done.add((int(row["layer"]), int(row["head"]), int(row["k"])))
    except Exception as e:
        print(f"  [warn] Could not read {csv_path}: {e}")
    return done


for model_key in MODEL_CONFIGS:
    iou_csv   = MODEL_CONFIGS[model_key]["out_csv"]
    out_csv   = INTERP_CONFIGS[model_key]
    model_name = MODEL_CONFIGS[model_key]["model_name"]

    print(f"\n{'='*60}")
    print(f"  Model: {model_key}  ({model_name})")
    print(f"  Reading IOU from: {iou_csv}")
    print(f"  Writing to:       {out_csv}")
    print(f"{'='*60}")

    # ── check Part 1 data ─────────────────────────────────────────────────────
    if not os.path.exists(iou_csv):
        print(f"  [SKIP] Part 1 CSV not found: {iou_csv}")
        continue

    iou_df = pd.read_csv(iou_csv)
    if iou_df.empty:
        print(f"  [SKIP] Part 1 CSV is empty.")
        continue

    # ── load model ────────────────────────────────────────────────────────────
    try:
        model_obj = AutoModel.from_pretrained(model_name, output_attentions=True)
        tokenizer_obj = AutoTokenizer.from_pretrained(model_name)
        model_obj.eval()
    except Exception as e:
        print(f"  [ERROR] Could not load model: {e}")
        traceback.print_exc()
        continue

    num_layers = model_obj.config.num_hidden_layers
    num_heads  = model_obj.config.num_attention_heads

    # ── build per-head program ranking from Part 1 ────────────────────────────
    # mean iou per (layer, head, program) across the 5 sentences
    head_rankings: dict = {}   # (layer, head) -> ordered list of Callable

    grouped = (
        iou_df.groupby(["layer", "head", "program_idx"])["iou_score"]
        .mean()
        .reset_index()
    )
    for (layer, head), grp in grouped.groupby(["layer", "head"]):
        sorted_grp = grp.sort_values("iou_score", ascending=False)
        ordered_fns = []
        for p_idx in sorted_grp["program_idx"]:
            fn = all_programs[int(p_idx)] if int(p_idx) < len(all_programs) else None
            if fn is not None:
                ordered_fns.append(fn)
        head_rankings[(int(layer), int(head))] = ordered_fns

    max_k = max(K_VALUES)
    total = num_layers * num_heads * len(K_VALUES)
    done_keys = load_done_interp(out_csv)
    print(f"  Already done rows: {len(done_keys)}  |  Total to do: {total}")

    write_header = not os.path.exists(out_csv) or os.path.getsize(out_csv) == 0
    errors = 0
    done_count = len(done_keys)
    t0 = time.time()

    with open(out_csv, "a", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        if write_header:
            writer.writerow(PART2_COLUMNS)
            fh.flush()

        for layer in range(num_layers):
            for head in range(num_heads):
                ranked_fns = head_rankings.get((layer, head), all_programs)

                for k in K_VALUES:
                    key = (layer, head, k)
                    if key in done_keys:
                        done_count += 1
                        continue

                    top_k_fns = ranked_fns[:k]
                    if not top_k_fns:
                        top_k_fns = all_programs[:k]

                    iou = linear_interp_iou(
                        model_obj, tokenizer_obj,
                        layer, head, top_k_fns, SAMPLE_SENTENCES,
                    )

                    # store indices, not names — use data/program_index_map.json to recover names
                    prog_idx_used = "|".join(str(all_programs.index(f)) for f in top_k_fns)
                    iou_out = round(iou, 6) if not np.isnan(iou) else iou
                    writer.writerow([layer, head, k, iou_out, prog_idx_used])
                    fh.flush()
                    done_count += 1

                    if np.isnan(iou):
                        errors += 1

                    if done_count % 200 == 0:
                        elapsed = time.time() - t0
                        pct = done_count / total * 100
                        eta = elapsed / done_count * (total - done_count) if done_count else 0
                        print(f"  [{model_key}] {done_count}/{total} ({pct:.1f}%) | "
                              f"elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | errors {errors}")

    del model_obj, tokenizer_obj
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    print(f"  DONE {model_key}.  Errors (NaN): {errors}")
    df_check = pd.read_csv(out_csv)
    print(f"  Rows saved: {len(df_check)}")

print("\n[PART 2 COMPLETE]")



  Model: tinyllama  (TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T)
  Reading IOU from: data/iou_scores_tinyllama.csv
  Writing to:       data/interpolation_k_tinyllama_3.csv
  Already done rows: 0  |  Total to do: 14080
  [tinyllama] 200/14080 (1.4%) | elapsed 13.7m | ETA 947.4m | errors 0
  [tinyllama] 400/14080 (2.8%) | elapsed 27.3m | ETA 934.8m | errors 0
  [tinyllama] 600/14080 (4.3%) | elapsed 40.7m | ETA 913.9m | errors 0
  [tinyllama] 800/14080 (5.7%) | elapsed 53.8m | ETA 892.4m | errors 0


KeyboardInterrupt: 

In [ ]:
# ── Final verification summary ────────────────────────────────────────────────
print("=== Part 1: IOU Scores ===")
for model_key, cfg in MODEL_CONFIGS.items():
    p = cfg["out_csv"]
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f"  {model_key}: {len(df)} rows | programs: {df['program'].nunique()} | "
              f"mean iou: {df['iou_score'].mean():.4f} | NaN: {df['iou_score'].isna().sum()}")
    else:
        print(f"  {model_key}: NOT FOUND at {p}")

print()
print("=== Part 2: Interpolation (k sweep) ===")
for model_key, p in INTERP_CONFIGS.items():
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f"  {model_key}: {len(df)} rows | k range: {df['k'].min()}..{df['k'].max()} | "
              f"mean iou: {df['iou_score'].mean():.4f} | NaN: {df['iou_score'].isna().sum()}")
    else:
        print(f"  {model_key}: NOT FOUND at {p}")
